# 店舗売上需要予測: 実験 #005 3大GBDT多様性アンサンブル (LGB + CatBoost + XGBoost with GPU)

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** RTX 4060 / Kaggle GPU をフル活用した **「LightGBM ＋ CatBoost ＋ XGBoost の 3大GBDT多様性アンサンブル」** ＋ **前年同期ラグ (Lag 364)** ＋ **ゼロ売上フィルター**
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **現在自己ベスト:** 実験 #004: **LB 0.45505 (277位)** 🏆

---

## 1. 3大GBDT多様性アンサンブルのメカニズム

単一モデルの限界を破り、決定木の「3大アルゴリズム」が持つ異なる予測特性をブレンドします：
1. **LightGBM (Leaf-wise):**
   - 最も深く分岐し、特売日や週末のスパイク（急増）需要を鋭く捉える。
2. **CatBoost (Symmetric Trees, GPU加速):**
   - 左右対称の決定木構造により過学習を強力に抑制し、安定したベースライン需要を滑らかに推計する。
3. **XGBoost (Depth-wise, GPU加速):**
   - 深さ優先の正則化木。両者の中間として汎化誤差を整える。

さらに、8月後半の新学期特需（2016年8月の学用品特需）を直接参照できるよう、**「ちょうど52週前の同じ曜日（364日前）の売上ラグ (`sales_lag_364`)」** を追加投入！

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    if os.path.exists('/kaggle/input'):
        input_contents = os.listdir('/kaggle/input')
        raise FileNotFoundError(f'/kaggle/input にデータがありません。現在の中身: {input_contents}。右上の [+ Add Input] から "store-sales-time-series-forecasting" を追加してください。')
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# 直近21日間に売上ゼロのペアを特定
recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))
print(f'直近21日間で売上ゼロのペア数: {len(zero_pairs)} / 1,782 ペア')

## 3. カレンダー ＆ 前年同期ラグ特徴量エンジニアリング

In [ ]:
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)
all_df['is_wage_day'] = ((all_df['day'] == 15) | (all_df['is_month_end'] == 1)).astype(int)
all_df['is_wage_day_after'] = ((all_df['day'] == 16) | (all_df['is_month_start'] == 1)).astype(int)

all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group = all_df.groupby(['store_nbr', 'family'])['log_sales']

# 前年同期（364日前＝ちょうど52週前の同じ曜日）を含むラグ設計
for lag in [16, 21, 28, 35, 364]:
    all_df[f'sales_lag_{lag}'] = group.shift(lag)

all_df['sales_lag_364'] = all_df['sales_lag_364'].fillna(all_df['sales_lag_28'])

lag16 = group.shift(16)
all_df['sales_rolling_mean_7'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
all_df['sales_rolling_mean_14'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)
all_df['sales_rolling_mean_28'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(28, min_periods=1).mean()
)

train_feat = all_df[all_df['date'] <= '2017-08-15'].copy()
test_feat = all_df[all_df['date'] >= '2017-08-16'].copy()

features = [
    'store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'dayofyear', 'weekofyear', 'is_month_start', 'is_month_end', 'is_wage_day', 'is_wage_day_after',
    'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'sales_lag_35', 'sales_lag_364',
    'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28'
]
print(f'総特徴量数: {len(features)} 列')

## 4. 3大GBDT (LGB + CatBoost + XGBoost with GPU) 多様性アンサンブル学習

In [ ]:
val_start_date = '2017-07-31'
val_end_date = '2017-08-15'
train_start_date = '2016-01-01'

trn_mask = (train_feat['date'] >= train_start_date) & (train_feat['date'] < val_start_date)
val_mask = (train_feat['date'] >= val_start_date) & (train_feat['date'] <= val_end_date)

families = train_df['family'].unique()

# GPU利用判定
try:
    test_cb = cb.CatBoostRegressor(iterations=1, task_type='GPU', verbose=0)
    test_cb.fit(np.random.randn(10, 2), np.random.randn(10))
    cb_task_type = 'GPU'
    xgb_device = 'cuda'
    print('⚡ GPU (CUDA) 加速が有効です！')
except Exception:
    cb_task_type = 'CPU'
    xgb_device = 'cpu'
    print('ℹ️ CPUモードで実行します。')

val_preds = []
val_trues = []
test_predictions = []

print('33商品ファミリーごとに 3大GBDTモデルを学習・ブレンド中...')
for fam in families:
    fam_trn_mask = trn_mask & (train_feat['family'] == fam)
    fam_val_mask = val_mask & (train_feat['family'] == fam)
    fam_tst_mask = (test_feat['family'] == fam)

    X_trn = train_feat.loc[fam_trn_mask, features].copy()
    y_trn = np.log1p(train_feat.loc[fam_trn_mask, 'sales'].values)
    X_v = train_feat.loc[fam_val_mask, features].copy()
    y_v = train_feat.loc[fam_val_mask, 'sales'].values
    X_tst = test_feat.loc[fam_tst_mask, features].copy()

    # 1. LightGBM
    X_trn_lgb, X_v_lgb, X_tst_lgb = X_trn.copy(), X_v.copy(), X_tst.copy()
    for df_lgb in [X_trn_lgb, X_v_lgb, X_tst_lgb]:
        df_lgb['store_nbr'] = df_lgb['store_nbr'].astype('category')
    lgb_model = lgb.LGBMRegressor(
        objective='regression', metric='rmse', learning_rate=0.05, num_leaves=31,
        feature_fraction=0.8, random_state=42, verbose=-1, n_estimators=500
    )
    lgb_model.fit(X_trn_lgb, y_trn, eval_set=[(X_v_lgb, np.log1p(y_v))], callbacks=[lgb.early_stopping(30, verbose=False)])
    lgb_v = lgb_model.predict(X_v_lgb)
    lgb_tst = lgb_model.predict(X_tst_lgb)

    # 2. CatBoost
    cb_model = cb.CatBoostRegressor(
        iterations=500, learning_rate=0.06, depth=6, loss_function='RMSE',
        task_type=cb_task_type, random_seed=42, verbose=0, early_stopping_rounds=30
    )
    cb_model.fit(X_trn, y_trn, eval_set=(X_v, np.log1p(y_v)), verbose=0)
    cb_v = cb_model.predict(X_v)
    cb_tst = cb_model.predict(X_tst)

    # 3. XGBoost
    xgb_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', device=xgb_device, random_state=42, early_stopping_rounds=30
    )
    xgb_model.fit(X_trn, y_trn, eval_set=[(X_v, np.log1p(y_v))], verbose=False)
    xgb_v = xgb_model.predict(X_v)
    xgb_tst = xgb_model.predict(X_tst)

    # 3大GBDT黄金ブレンド (50% LGB + 30% CB + 20% XGB)
    ens_pred_v = np.clip(np.expm1(0.50 * lgb_v + 0.30 * cb_v + 0.20 * xgb_v), 0, None)
    ens_pred_tst = np.clip(np.expm1(0.50 * lgb_tst + 0.30 * cb_tst + 0.20 * xgb_tst), 0, None)

    # ゼロ売上フィルター
    tst_stores = test_feat.loc[fam_tst_mask, 'store_nbr'].values
    for i, st in enumerate(tst_stores):
        if (st, fam) in zero_pairs:
            ens_pred_tst[i] = 0.0

    val_preds.extend(ens_pred_v)
    val_trues.extend(y_v)

    fam_test_df = test_feat.loc[fam_tst_mask, ['id']].copy()
    fam_test_df['sales'] = ens_pred_tst
    test_predictions.append(fam_test_df)

ens_rmsle = np.sqrt(mean_squared_error(np.log1p(val_trues), np.log1p(val_preds)))
print('=' * 70)
print(f'  🏆 [実験 #005] 3大GBDT多様性アンサンブル 手元検証 RMSLE: {ens_rmsle:.5f}')
print('=' * 70)

## 5. テストデータ予測 & 提出ファイル作成 (id順ソート)

In [ ]:
# 全ファミリーの予測を結合し、id 順に整列
final_sub = pd.concat(test_predictions, ignore_index=True)
final_sub = final_sub.sort_values('id').reset_index(drop=True)
final_sub.to_csv('submission.csv', index=False)

print('提出用 submission.csv を作成しました（id順ソート完了）！')
print(f'行数: {len(final_sub):,} 行')
print(final_sub.head(10))
print(final_sub.describe())